In [1]:
from src.dataset_manager import DatasetManager, store_dataframe_csv

# 1. Reconstruir dataset base corregido
df = DatasetManager.get_base_dataset()

# Verificar corrección
train_motors = df[df['evento'].isin([0,1])]
eventos_por_motor = df.groupby('unit_number')['evento'].sum()
print("Eventos por motor (debe ser 0 o 1):")
print(eventos_por_motor.value_counts())

# 2. Limpiar columnas constantes
df_clean, stats = DatasetManager.clean_dataset(df)

# 3. Guardar metadata
metadata = DatasetManager.generate_metadata(df_clean)
store_dataframe_csv(metadata, 'metadata', 'data')
print(f"\nMetadata guardada: {len(metadata)} motores")

# 4. Guardar CSV por motor
for unit_id in df_clean['unit_number'].unique():
    motor_df = df_clean[df_clean['unit_number'] == unit_id].drop(
        columns=['unit_number']
    ).reset_index(drop=True)
    store_dataframe_csv(motor_df, f'data_motor_{unit_id}', 'data/clean')

print(f"CSVs generados: {df_clean['unit_number'].nunique()} motores")

Eventos por motor (debe ser 0 o 1):
evento
1    100
0    100
Name: count, dtype: int64
Zero-variance columns detected: 8
Columns removed: ['op_setting_3', 'T2', 'P2', 'P15', 'epr', 'farB', 'Nf_dmd', 'PCNf_dmd']

Metadata guardada: 200 motores
CSVs generados: 200 motores


In [3]:
# Guardar dataset unificado completo (sin filtrar)
store_dataframe_csv(df, 'dataset_unificado', 'data')
print("Dataset unificado guardado")

# Guardar también el limpio unificado por si acaso
store_dataframe_csv(df_clean, 'dataset_limpio', 'data')
print("Dataset limpio guardado")

Dataset unificado guardado
Dataset limpio guardado


In [2]:
import pandas as pd

# Verificar un motor de train — debe tener evento=1 solo en el último ciclo
motor_train = pd.read_csv('data/clean/data_motor_1.csv')
print("Motor 1 (train):")
print(f"  Filas: {len(motor_train)}")
print(f"  Suma evento: {motor_train['evento'].sum()} (debe ser 1)")
print(f"  Evento en último ciclo: {motor_train['evento'].iloc[-1]} (debe ser 1)")
print(f"  Evento en primer ciclo: {motor_train['evento'].iloc[0]} (debe ser 0)")
print(f"  RUL último ciclo: {motor_train['RUL'].iloc[-1]} (debe ser 0)")

# Verificar un motor de test — debe tener evento=0 en todos los ciclos
motor_test = pd.read_csv('data/clean/data_motor_101.csv')
print("\nMotor 101 (test/censurado):")
print(f"  Filas: {len(motor_test)}")
print(f"  Suma evento: {motor_test['evento'].sum()} (debe ser 0)")
print(f"  RUL último ciclo: {motor_test['RUL'].iloc[-1]} (debe ser > 0)")

Motor 1 (train):
  Filas: 192
  Suma evento: 1 (debe ser 1)
  Evento en último ciclo: 1 (debe ser 1)
  Evento en primer ciclo: 0 (debe ser 0)
  RUL último ciclo: 0 (debe ser 0)

Motor 101 (test/censurado):
  Filas: 31
  Suma evento: 0 (debe ser 0)
  RUL último ciclo: 112 (debe ser > 0)


In [4]:
import pandas as pd

metadata = pd.read_csv('data/metadata.csv')

# Debe haber exactamente 100 motores con event=1 y 100 con event=0
print(metadata['event'].value_counts())

# Verificar motor 1 específicamente
print(metadata[metadata['unit_number'] == 1])

event
1    100
0    100
Name: count, dtype: int64
   unit_number  max_cycles  n_settings  n_sensores  event
0            1         192           3          21      1


In [1]:
from src.dataset_manager import DatasetManager
from src.models.cox_frailty import CoxFrailty
from src.training_manager import GGSTrainingManager

m_train, m_test = DatasetManager.split_dataset()

Training = GGSTrainingManager(
    model=CoxFrailty(),
    list_ids=m_train
)

param_grid = {
    'distribution':         ['gamma', 'gaussian'],
    'confidence_threshold': [0.5, 0.7, 0.9],
    'clipping_threshold':   [110, 115, 120, 125]
}

ggs = Training.group_grid_search(
    param_grid=param_grid,
    n_folds=5,
    silence=True
)

display(Training.get_ggs_results(top_n=10))

/home/jdani/proyects/Premant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training engines: 140
Test engines: 60
Starting Grid Search: 24 configs × 5 folds = 120 tasks


GGS progress:   0%|          | 0/24 [00:00<?, ?it/s]

In addition: Warning messages:
1: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
 2: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
 3: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
 In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6 7
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6 7
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6 7
Error in frailty.brent(sqrt(x), y, lower = 0) : 
  Ties for max(y), I surrender
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inne

GGS progress:   4%|▍         | 1/24 [02:44<1:03:12, 164.91s/it]

...

In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6 7
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6


GGS progress:  96%|█████████▌| 23/24 [52:03<02:02, 122.34s/it]

In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6 7
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5
In addition: Warning message:
In coxpenal.fit(X, Y, istrat, offset, init = init, control, weights = weights,  :
  Inner loop failed to coverge for iterations 1 2 3 4 5 6


GGS progress: 100%|██████████| 24/24 [54:07<00:00, 135.30s/it]


,distribution,confidence_threshold,clipping_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE,Success
0,gaussian,0.5,110,1947.686673,0.5,22.483457,39.667066,1
1,gaussian,0.7,110,1947.686673,0.5,22.483457,39.667066,1
2,gaussian,0.9,110,1947.686673,0.5,22.483457,39.667066,1
3,gamma,0.5,110,1974.741332,0.5,22.709868,39.866962,1
4,gamma,0.7,110,1974.741332,0.5,22.709868,39.866962,1
5,gamma,0.9,110,1974.741332,0.5,22.709868,39.866962,1
6,gaussian,0.5,115,3211.491184,0.5,24.791312,42.540621,1
7,gaussian,0.7,115,3211.491184,0.5,24.791312,42.540621,1
8,gaussian,0.9,115,3211.491184,0.5,24.791312,42.540621,1
9,gamma,0.5,115,3256.099294,0.5,25.037064,42.754949,1


In [2]:
import warnings
warnings.filterwarnings('ignore')

from src.models.cox_frailty import CoxFrailty
from src.mad_scaler import MADScaler
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline

X, y_surv, y_metrics, groups = Training.get_training_data()

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(iter(gkf.split(X, y_surv, groups)))

model = CoxFrailty(distribution='gamma')
pipeline = Pipeline([('scaler', MADScaler()), ('model', model)])
pipeline.set_output(transform='pandas')

warnings.filterwarnings('error', category=RuntimeWarning)
try:
    pipeline.fit(X.iloc[train_idx], y_surv[train_idx], model__groups=groups[train_idx])
    print("is_fitted:", model.is_fitted_)
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

ERROR: RuntimeWarning: Fit failed for distribution='gamma', maxit=300: RuntimeError: coxph did not complete any iterations. Check the data or try a different distribution.


In [3]:
import warnings
warnings.filterwarnings('ignore')

from src.models.cox_frailty import CoxFrailty
from src.mad_scaler import MADScaler
from sklearn.model_selection import GroupKFold

X, y_surv, y_metrics, groups = Training.get_training_data()

gkf = GroupKFold(n_splits=5)
train_idx, _ = next(iter(gkf.split(X, y_surv, groups)))

X_fold = X.iloc[train_idx]
y_fold = y_surv[train_idx]
g_fold = groups[train_idx]

scaler = MADScaler()
X_scaled = scaler.fit_transform(X_fold)

# Ver exactamente qué llega a coxph
evento = y_fold['evento'].astype(int)
t_start = y_fold['t_start'].astype(float)
t_stop = y_fold['t_stop'].astype(float)

print(f"Filas: {len(X_fold)}")
print(f"Motores únicos: {len(set(g_fold.tolist()))}")
print(f"Eventos totales: {evento.sum()}")
print(f"t_start range: {t_start.min()} - {t_start.max()}")
print(f"t_stop range: {t_stop.min()} - {t_stop.max()}")
print(f"Eventos en último ciclo de cada motor:")
import pandas as pd
df_check = pd.DataFrame({'motor': g_fold, 'evento': evento, 't_stop': t_stop})
print(df_check.groupby('motor')['evento'].sum().value_counts())

Filas: 19062
Motores únicos: 112
Eventos totales: 0
t_start range: 0.0 - 286.0
t_stop range: 1.0 - 287.0
Eventos en último ciclo de cada motor:
evento
0    112
Name: count, dtype: int64
